In [1]:
import os
import zipfile
import shutil
import pandas as pd
from pathlib import Path
from google.colab import files

In [2]:
events = pd.read_csv("events_step3_ok_updated.csv")
price = pd.read_csv("4_price.csv")

print("events rows:", len(events))
print("price rows:", len(price))
display(events.head())
display(price.head())

events rows: 2265
price rows: 191288


/tmp/ipykernel_9894/1261508992.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  price = pd.read_csv("4_price.csv")


,ticker,cik,filing_date,filing_type,accession_number,year,quarter,cik_nolead,acc_nodash,mda_path,text_length_words,mda_status,primary_doc
0,AAPL,320193,2019-01-30,10-Q,0000320193-19-000010,2019,1,320193,32019319000010,AAPL_20190130_10-Q_000032019319000010.txt,8130,OK,a10-qq1201912292018.htm
1,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2,320193,32019319000066,AAPL_20190501_10-Q_000032019319000066.txt,8286,OK,a10-qq220193302019.htm
2,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3,320193,32019319000076,AAPL_20190731_10-Q_000032019319000076.txt,8166,OK,a10-qq320196292019.htm
3,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1,320193,32019320000010,AAPL_20200129_10-Q_000032019320000010.txt,3926,OK,a10-qq1202012282019.htm
4,AAPL,320193,2020-05-01,10-Q,0000320193-20-000052,2020,2,320193,32019320000052,AAPL_20200501_10-Q_000032019320000052.txt,5146,OK,a10-qq220203282020.htm


,PERMNO,date,TICKER,NWPERM,PRC,RET,CFACPR
0,10104,2019-01-02,ORCL,NaN,45.22,0.001550,1.0
1,10104,2019-01-03,ORCL,NaN,44.78,-0.009730,1.0
2,10104,2019-01-04,ORCL,NaN,46.71,0.043100,1.0
3,10104,2019-01-07,ORCL,NaN,47.45,0.015842,1.0
4,10104,2019-01-08,ORCL,NaN,47.88,0.009062,1.0


In [4]:
print("events columns:")
print(events.columns.tolist())

print("\nprice columns:")
print(price.columns.tolist())

events columns:
['ticker', 'cik', 'filing_date', 'filing_type', 'accession_number', 'year', 'quarter', 'cik_nolead', 'acc_nodash', 'mda_path', 'text_length_words', 'mda_status', 'primary_doc']

price columns:
['PERMNO', 'date', 'TICKER', 'NWPERM', 'PRC', 'RET', 'CFACPR']


In [5]:
events.columns = [c.strip() for c in events.columns]
price.columns = [c.strip() for c in price.columns]

# Standardise ticker column names
if "TICKER" in price.columns and "ticker" not in price.columns:
    price = price.rename(columns={"TICKER": "ticker"})

if "Ticker" in price.columns and "ticker" not in price.columns:
    price = price.rename(columns={"Ticker": "ticker"})

if "ticker" not in events.columns and "TICKER" in events.columns:
    events = events.rename(columns={"TICKER": "ticker"})

print("Cleaned events columns:")
print(events.columns.tolist())

print("\nCleaned price columns:")
print(price.columns.tolist())

Cleaned events columns:
['ticker', 'cik', 'filing_date', 'filing_type', 'accession_number', 'year', 'quarter', 'cik_nolead', 'acc_nodash', 'mda_path', 'text_length_words', 'mda_status', 'primary_doc']

Cleaned price columns:
['PERMNO', 'date', 'ticker', 'NWPERM', 'PRC', 'RET', 'CFACPR']


In [6]:
events["ticker"] = events["ticker"].astype(str).str.strip().str.upper()
price["ticker"] = price["ticker"].astype(str).str.strip().str.upper()

print("Unique tickers in events:", events["ticker"].nunique())
print("Unique tickers in price:", price["ticker"].nunique())

Unique tickers in events: 124
Unique tickers in price: 132


In [8]:
event_tickers = set(events["ticker"].dropna())
price_tickers = set(price["ticker"].dropna())

missing_tickers = sorted(event_tickers - price_tickers)

print("Tickers needed from Step 3:", len(event_tickers))
print("Tickers available in 4_price.csv:", len(price_tickers))
print("Missing tickers from 4_price.csv:", len(missing_tickers))

if missing_tickers:
    print("Missing ticker list:")
    for t in missing_tickers:
        print(t)

Tickers needed from Step 3: 124
Tickers available in 4_price.csv: 132
Missing tickers from 4_price.csv: 1
Missing ticker list:
BRK.B


In [9]:
events["ticker"] = events["ticker"].astype(str).str.strip().str.upper()
price["ticker"] = price["ticker"].astype(str).str.strip().str.upper()

print("Unique tickers in events:", events["ticker"].nunique())
print("Unique tickers in price:", price["ticker"].nunique())

Unique tickers in events: 124
Unique tickers in price: 132


In [10]:
event_tickers = set(events["ticker"].dropna())
price_tickers = set(price["ticker"].dropna())

# exclude BRK.B because you already confirmed it is unavailable in WRDS
event_tickers_for_step4 = set(t for t in event_tickers if t != "BRK.B")

missing_tickers = sorted(event_tickers_for_step4 - price_tickers)

print("Tickers needed from Step 3:", len(event_tickers))
print("Tickers used for Step 4 after excluding BRK.B:", len(event_tickers_for_step4))
print("Tickers available in 4_price.csv:", len(price_tickers))
print("Missing tickers from 4_price.csv after excluding BRK.B:", len(missing_tickers))

if missing_tickers:
    print("Missing ticker list:")
    for t in missing_tickers:
        print(t)

Tickers needed from Step 3: 124
Tickers used for Step 4 after excluding BRK.B: 123
Tickers available in 4_price.csv: 132
Missing tickers from 4_price.csv after excluding BRK.B: 0


In [11]:
price["date"] = pd.to_datetime(price["date"], errors="coerce")
price = price.dropna(subset=["date"]).copy()

events["filing_date"] = pd.to_datetime(events["filing_date"], errors="coerce")

price.columns = [c.strip() for c in price.columns]

display(price.head())
print("Rows after date cleaning:", len(price))

,PERMNO,date,ticker,NWPERM,PRC,RET,CFACPR
0,10104,2019-01-02,ORCL,NaN,45.22,0.001550,1.0
1,10104,2019-01-03,ORCL,NaN,44.78,-0.009730,1.0
2,10104,2019-01-04,ORCL,NaN,46.71,0.043100,1.0
3,10104,2019-01-07,ORCL,NaN,47.45,0.015842,1.0
4,10104,2019-01-08,ORCL,NaN,47.88,0.009062,1.0


Rows after date cleaning: 191288


In [12]:
if "adj_close" not in price.columns:
    if {"PRC", "CFACPR"}.issubset(price.columns):
        price["adj_close"] = price["PRC"].abs() / price["CFACPR"]
    else:
        raise ValueError("adj_close missing, and PRC/CFACPR not available to rebuild it.")

if "return" not in price.columns:
    if "RET" in price.columns:
        price = price.rename(columns={"RET": "return"})
    else:
        raise ValueError("return missing, and RET not available to rebuild it.")

print("Columns available:")
print(price.columns.tolist())

Columns available:
['PERMNO', 'date', 'ticker', 'NWPERM', 'PRC', 'return', 'CFACPR', 'adj_close']


In [14]:
price_updated = price.loc[
    price["ticker"].isin(event_tickers_for_step4)
].copy()

print("Original Step 4 rows:", len(price))
print("Updated Step 4 rows:", len(price_updated))
print("Updated Step 4 unique tickers:", price_updated["ticker"].nunique())

Original Step 4 rows: 191288
Updated Step 4 rows: 182815
Updated Step 4 unique tickers: 123


In [15]:
preferred_cols = ["PERMNO", "date", "ticker", "adj_close", "return"]

existing_cols = [c for c in preferred_cols if c in price_updated.columns]
price_updated = price_updated[existing_cols].copy()

price_updated = price_updated.sort_values(["ticker", "date"]).reset_index(drop=True)

display(price_updated.head())
print("Final Step 4 master rows:", len(price_updated))

,PERMNO,date,ticker,adj_close,return
0,14593,2019-01-02,AAPL,39.480000,0.001141
1,14593,2019-01-03,AAPL,35.547500,-0.099607
2,14593,2019-01-04,AAPL,37.064997,0.042689
3,14593,2019-01-07,AAPL,36.982498,-0.002226
4,14593,2019-01-08,AAPL,37.687500,0.019063


Final Step 4 master rows: 182815


In [16]:
STEP4_UPDATED_CSV = "/content/4_price_updated.csv"
price_updated.to_csv(STEP4_UPDATED_CSV, index=False)

print("Saved:", STEP4_UPDATED_CSV)

Saved: /content/4_price_updated.csv


In [17]:
PRICE_DIR = Path("/content/04_prices_updated")
PRICE_DIR.mkdir(parents=True, exist_ok=True)

for old_file in PRICE_DIR.glob("*.csv"):
    old_file.unlink()

for ticker, group in price_updated.groupby("ticker"):
    group = group.sort_values("date")
    group.to_csv(PRICE_DIR / f"{ticker}.csv", index=False)

print("Ticker files created:", len(list(PRICE_DIR.glob('*.csv'))))

Ticker files created: 123


In [18]:
STEP4_UPDATED_ZIP = "/content/ticker_price_files_filtered_updated.zip"

with zipfile.ZipFile(STEP4_UPDATED_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in PRICE_DIR.glob("*.csv"):
        zipf.write(file, arcname=file.name)

print("Saved:", STEP4_UPDATED_ZIP)

Saved: /content/ticker_price_files_filtered_updated.zip


In [19]:
with zipfile.ZipFile(STEP4_UPDATED_ZIP, "r") as zipf:
    zip_names = set(zipf.namelist())

csv_tickers = set(price_updated["ticker"].dropna().unique())
zip_tickers = set([Path(x).stem for x in zip_names])

print("Unique tickers in updated Step 4 master CSV:", len(csv_tickers))
print("Ticker files in ZIP:", len(zip_tickers))
print("Missing ticker files in ZIP:", len(csv_tickers - zip_tickers))
print("Extra ticker files in ZIP:", len(zip_tickers - csv_tickers))

Unique tickers in updated Step 4 master CSV: 123
Ticker files in ZIP: 123
Missing ticker files in ZIP: 0
Extra ticker files in ZIP: 0


In [20]:
from google.colab import files

files.download(STEP4_UPDATED_CSV)
files.download(STEP4_UPDATED_ZIP)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>